In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [66]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# 🏭 Manufacturing Quality Prediction using Polynomial Regression

**🇹🇷 Türkçe:**
Bu projenin amacı, endüstriyel üretim sürecindeki sensör verilerini (Sıcaklık, Basınç, Materyal Füzyon Metrikleri vb.) kullanarak üretilen ürünün **Kalite Derecesini (Quality Rating)** tahmin etmektir. Verideki değişkenler arasındaki ilişkinin doğrusal (linear) olmama ihtimaline karşı modelimizi **Polinom Regresyon (Polynomial Regression)** ile güçlendireceğiz.

**🇬🇧 English:**
The objective of this project is to predict the **Quality Rating** of manufactured products based on industrial sensor data (Temperature, Pressure, Material Fusion Metrics, etc.). Since the relationship between the features and the target variable might be non-linear, we will enhance our predictive model using **Polynomial Regression**.

In [67]:
df = pd.read_csv("../../Data/manufacturing.csv")

In [68]:
df.head()

,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric,Quality Rating
0,209.762701,8.050855,1688.769167,44522.217074,9.229576e+06,99.999971
1,243.037873,15.812068,3842.931469,63020.764997,1.435537e+07,99.985703
2,220.552675,7.843130,1729.823314,49125.950249,1.072839e+07,99.999758
3,208.976637,23.786089,4970.736918,57128.881547,9.125702e+06,99.999975
4,184.730960,15.797812,2918.345014,38068.201283,6.303792e+06,100.000000


In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3957 entries, 0 to 3956
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Temperature (°C)                3957 non-null   float64
 1   Pressure (kPa)                  3957 non-null   float64
 2   Temperature x Pressure          3957 non-null   float64
 3   Material Fusion Metric          3957 non-null   float64
 4   Material Transformation Metric  3957 non-null   float64
 5   Quality Rating                  3957 non-null   float64
dtypes: float64(6)
memory usage: 185.6 KB


In [70]:
df.describe()

,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric,Quality Rating
count,3957.000000,3957.000000,3957.000000,3957.000000,3.957000e+03,3957.000000
mean,200.034704,14.815558,2955.321308,48127.183128,1.003645e+07,96.260179
std,58.135717,5.772040,1458.224940,23812.213513,7.599356e+06,12.992262
min,100.014490,5.003008,513.706875,10156.971955,9.999462e+05,1.000000
25%,150.871296,9.692984,1798.247303,27626.929091,3.433810e+06,99.941129
50%,198.603371,14.832557,2678.277782,44611.452164,7.833390e+06,99.999997
75%,251.366552,19.749680,3929.058261,67805.443846,1.588251e+07,100.000000
max,299.992804,24.999132,7365.018714,103756.181544,2.699783e+07,100.000000


In [71]:
df.isnull().sum()

Temperature (°C)                  0
Pressure (kPa)                    0
Temperature x Pressure            0
Material Fusion Metric            0
Material Transformation Metric    0
Quality Rating                    0
dtype: int64

## ⚙️ Veri Ön İşleme ve Ölçeklendirme / Data Preprocessing & Scaling

**🇹🇷 Türkçe:**
Veri setimizi Eğitim (Train) ve Test olarak ayırdıktan sonra **StandardScaler** uyguluyoruz. Makine öğrenmesi modellerinde basınç (örneğin 15 kPa) ve metrik değerleri (örneğin 14.000.000) gibi çok farklı ölçeklerdeki sayıların modeli yanıltmasını engellemek için tüm değişkenleri ortak bir Z-Skor ölçeğine (Ortalama: 0, Standart Sapma: 1) getiriyoruz.

**🇬🇧 English:**
After splitting our dataset into Training and Test sets, we apply **StandardScaler**. To prevent features with vastly different scales (e.g., Pressure at 15 kPa vs. Transformation Metrics at 14,000,000) from dominating the model, we standardize all features to a common Z-Score scale (Mean: 0, Standard Deviation: 1).

In [72]:
scaler = StandardScaler()

In [73]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [74]:
X


,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric
0,209.762701,8.050855,1688.769167,44522.217074,9.229576e+06
1,243.037873,15.812068,3842.931469,63020.764997,1.435537e+07
2,220.552675,7.843130,1729.823314,49125.950249,1.072839e+07
3,208.976637,23.786089,4970.736918,57128.881547,9.125702e+06
4,184.730960,15.797812,2918.345014,38068.201283,6.303792e+06
...,...,...,...,...,...
3952,156.811578,21.794290,3417.596965,34941.963896,3.855501e+06
3953,197.850406,8.291704,1640.516924,39714.857236,7.744742e+06
3954,241.357144,16.391910,3956.304672,62657.690952,1.405957e+07
3955,209.040239,23.809936,4977.234763,57195.985528,9.134036e+06


In [75]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 15)

In [76]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 🧮 Polinom Özellik Mühendisliği / Polynomial Feature Engineering

**🇹🇷 Türkçe:**
Değişkenlerin sadece kendilerini değil, birbirleriyle olan etkileşimlerini (örneğin Sıcaklık ve Basıncın karesel veya çapraz çarpım etkileri) modele anlatabilmek için `PolynomialFeatures(degree=2)` kullanıyoruz. Bu işlem, elimizdeki 5 bağımsız değişkeni 21 boyuta çıkararak modelin karmaşık eğrileri yakalamasını sağlar.

**🇬🇧 English:**
To capture the complex interactions and non-linear relationships between variables (e.g., the squared or cross-product effects of Temperature and Pressure), we use `PolynomialFeatures(degree=2)`. This transforms our initial 5 features into 21 dimensions, allowing the model to fit non-linear curves accurately.

In [77]:
poly = PolynomialFeatures(degree = 2)

In [78]:
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

In [79]:
X_train_poly

array([[ 1.00000000e+00,  5.16006193e-01, -4.70083140e-01, ...,
         7.61972685e-02,  7.78888892e-02,  7.96180647e-02],
       [ 1.00000000e+00,  1.28170161e+00,  6.04369979e-01, ...,
         1.97114235e+00,  1.97495710e+00,  1.97877923e+00],
       [ 1.00000000e+00, -3.02253952e-01, -1.60187153e+00, ...,
         3.80094392e-01,  3.22172353e-01,  2.73076970e-01],
       ...,
       [ 1.00000000e+00,  8.59472018e-01, -1.33205957e+00, ...,
         3.84320738e-01,  4.57406981e-01,  5.44392029e-01],
       [ 1.00000000e+00,  1.51743229e+00, -9.63189328e-02, ...,
         2.53591803e+00,  2.92431584e+00,  3.37220015e+00],
       [ 1.00000000e+00, -2.74456223e-01,  1.70928215e+00, ...,
         1.09996044e-03, -1.66209492e-02,  2.51150807e-01]],
      shape=(3165, 21))

In [80]:
regression = LinearRegression()

In [ ]:
regression.fit(X_train_poly, y_train)

In [61]:
y_pred = regression.predict(X_test_poly)

In [62]:
r2_score(y_test, y_pred)

0.9272003108305875